# LandIQ 2018 — Filter Non-Tomato Negatives (binary: tomato vs no tomato)

Creates `landiq_non_tomato_<YEAR>.gpkg` by:
1. Excluding any polygon where ANY of `CROPTYP1/2/3` contains tomato codes (default `T15`/`T26`).
2. Sampling negatives to match the tomato count.
3. Balancing negatives across a coarse DWR group derived from the leading crop code letter.

Output is written to `data/derived/landiq_non_tomato/` (gitignored).


In [ ]:
from __future__ import annotations

import geopandas as gpd
import pandas as pd
import yaml

from src.landiq.filter_non_tomato import filter_non_tomatoes_from_landiq_config
from src.utils.paths import REPO_ROOT, landiq_non_tomato_gpkg_path, resolve_landiq_shapefile_path

cfg_path = REPO_ROOT / "configs" / "paths.local.yaml"
if not cfg_path.is_file():
    cfg_path = REPO_ROOT / "configs" / "paths.example.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
lc = cfg["landiq"]

in_path = resolve_landiq_shapefile_path(cfg, REPO_ROOT)
gdf = gpd.read_file(in_path)

# Drop huge negative polygons before sampling (prevents city/forest-sized shapes).
# Shape_STAr is m^2 in this LandIQ layer.
MAX_AREA_M2 = 10_000_000.0  # 10 km^2 cap (tune if needed)
MAX_AREA_Q = 0.995  # also drop top 0.5% by area among non-tomato

# target_n defaults to equal-to-tomato-count inside the function.
non_gdf = filter_non_tomatoes_from_landiq_config(
    gdf,
    lc,
    seed=42,
    max_area_m2=MAX_AREA_M2,
    max_area_quantile=MAX_AREA_Q,
)

out_gpkg = landiq_non_tomato_gpkg_path(cfg, REPO_ROOT)
out_gpkg.parent.mkdir(parents=True, exist_ok=True)
non_gdf.to_file(out_gpkg, driver="GPKG")

print("Input:", in_path)
print("Tomato values:", lc.get("tomato_values"))
print("CROPTYP columns:", lc.get("crop_columns") or [lc.get("crop_column")])
print("Huge polygon filter: max_area_m2=", MAX_AREA_M2, "| max_area_quantile=", MAX_AREA_Q)

# Quick sanity: show largest areas in the sampled negatives (if Shape_STAr exists)
if "Shape_STAr" in non_gdf.columns:
    s = pd.to_numeric(non_gdf["Shape_STAr"], errors="coerce")
    print("Sampled negative area (m^2) — max:", float(s.max()), "| p99:", float(s.quantile(0.99)))

print("Wrote non-tomato polygons:", len(non_gdf))
print("Output:", out_gpkg.resolve())
